Мы скомпоновали все вакансии в одну модель данных (13.000+ вакансий)

Но есть проблема: часть вакансий - мусор. Их нужно отсечь

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

**Baseline: TF-IDF + IsolationForest | DBSCAN | LOF**

Использовать TF-IDF будем на названиях вакансий

Для этого также предобработаем названия

In [2]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

Уберем стоп-слова (не несут смысловой нагрузки) + лемматизация

In [3]:
stop_words = set(stopwords.words('russian'))
punctuation = set(string.punctuation)

In [4]:
import string
import pymorphy3

In [5]:
def preprocess_and_lemmatize(text: str) -> str:
    """
    Токенизация + фильтрация (NLTK) + лемматизация (Pymorphy3).
    Возвращает строку из лемм.
    """
    words = word_tokenize(text.lower())
    filtered_words = [
        word for word in words
        if word not in stop_words and word not in punctuation
    ]

    lemmatized_words = []
    for word in filtered_words:
        parsed = morph.parse(word)[0]
        lemma = parsed.normal_form
        lemmatized_words.append(lemma)

    return " ".join(lemmatized_words)

In [6]:
morph = pymorphy3.MorphAnalyzer()

Импортируем данные

In [42]:
df = pd.read_csv("data/vacancies.csv", encoding='utf-16')

In [43]:
df.head()

,title,salary_from,salary_to,salary_currency,area,experience_from,experience_to,company,skills,desc
0,Программист 1С (Middle),200000.0,NaN,RUB,екатеринбург,3.0,6.0,ООО Практик-Центр,"ERP-системы на базе 1С, 1С: Управление Торговл...",Полная занятость\nОформление : Трудовой догово...
1,Sales & Recruitment Manager,800.0,1300.0,EUR,армения,1.0,3.0,Enline,Английский — B2 — Средне-продвинутый,Выплаты: два раза в месяц\nПолная занятость\nФ...
2,Специалист по автоматизации бизнес-процессов и...,120000.0,150000.0,RUB,армавир,1.0,3.0,ООО Горница,NaN,Полная занятость\nОформление : Трудовой догово...
3,Программист АСУТП,150000.0,NaN,RUB,село борщёвка,1.0,3.0,"ООО «РУСАГРО», Мясное бизнес-направление",NaN,Выплаты: два раза в месяц\nПолная занятость\nГ...
4,Методолог отдела обучения,65000.0,NaN,RUB,самара,1.0,3.0,ВИТА. Офис,принимать решения и за их результаты отвечать ...,Выплаты: два раза в месяц\nПолная занятость\nО...


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
df['title_preprocessed'] = df['title'].apply(preprocess_and_lemmatize)

In [12]:
vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=None,
    sublinear_tf=True,
    norm="l2",
    smooth_idf=True,
    use_idf=True
)

In [14]:
df['title_preprocessed'].isna().sum()

0

In [19]:
tf_matrix = vectorizer.fit_transform(df['title_preprocessed'])

1. IsolationForest

In [22]:
from sklearn.ensemble import IsolationForest

In [37]:
forest = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    max_samples="auto",
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)

In [38]:
forest.fit(tf_matrix)

IsolationForest(contamination=0.01, n_estimators=200, n_jobs=-1,
                random_state=42)

In [39]:
predictions = forest.predict(tf_matrix)
scores = forest.decision_function(tf_matrix)

df["is_anomaly"] = predictions == -1
df["anomaly_score"] = scores

In [ ]:
df[df['is_anomaly'] == True].to_excel("data/anomalies.xlsx", index=False)

Результат неудовлетворительный: в выборку аномальных вакансий попали, например:
- Senior Fullstack-разработчик (Go/Java + Angular/React)
- Middle++ / Senior Go Backend Developer
- Начальник службы информационной безопасности В архиве с 11 октября 2026
- C#/.NET Backend Developer (Middle+/Senior) В архиве с 16 октября 2026
- Middle+/Senior Backend Developer (Node.js) В архиве с 15 октября 2026

Попробуем другой план:
1. Разметим выборку
2. Обучим классфикатор
3. Посмотреть на метрики (Precision, F1)

Если качество будет удовлетворять, то применить на весь датасет

С формированием обучающей выборки используем ИИ-агента

In [75]:
X = pd.read_excel("data/classified_sample_1000_final.xlsx", sheet_name="Классификация 1000")

In [76]:
X = X[['Название вакансии', 'Классификация']]

In [77]:
X['Классификация'] = X['Классификация'].apply(lambda x: 1 if x == 'IT' else 0)

In [78]:
X.head()

,Название вакансии,Классификация
0,Менеджер/оператор 1С,1
1,Руководитель технического отдела,1
2,Координатор проектов в ИТ направление,1
3,Инженер группы автоматизации,1
4,Продуктовый аналитик / SQL-аналитик,1


In [79]:
X['Классификация'].value_counts()

Классификация
0    575
1    425
Name: count, dtype: int64

In [80]:
X["preprocessed"] = (
    X["Название вакансии"].apply(preprocess_and_lemmatize)
)

In [81]:
X.columns = ["title", "label", "title_preprocessed"]

In [82]:
y = X['label']
X = X['title_preprocessed']


In [83]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

In [84]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42,
    stratify=y
)

In [94]:
pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True,
            norm="l2"
        )
    ),
    (
        "svm",
        SVC(
            kernel="linear",
            C=1.0,
            class_weight="balanced"
        )
    )
])

In [95]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('svm', SVC(class_weight='balanced', kernel='linear'))])

In [96]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)


In [97]:
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8272727272727273

Classification report:
              precision    recall  f1-score   support

           0       0.86      0.84      0.85       190
           1       0.79      0.81      0.80       140

    accuracy                           0.83       330
   macro avg       0.82      0.83      0.82       330
weighted avg       0.83      0.83      0.83       330


Confusion matrix:
[[159  31]
 [ 26 114]]


In [99]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "svm__C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_micro",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)
print("Лучший F1:", grid_search.best_score_)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Лучшие параметры: {'svm__C': 0.1}
Лучший F1: 0.8104477611940298


In [100]:
results = pd.DataFrame({
    "title": X_test,
    "real": y_test,
    "predicted": y_pred,
})

errors = results[results["real"] != results["predicted"]]

In [103]:
errors

,title,real,predicted
434,инженер-геодезист промышленный строительство,0,1
246,руководитель проект bitrix24,1,0
695,инженер служба эксплуатация информационный сис...,0,1
128,специалист разработка unity,1,0
193,стажёр группа управление выручка доходность rg...,0,1
68,менеджер первый линия отдел забота,0,1
122,технический директор,1,0
455,главный специалист аналитический департамент с...,1,0
730,мастер ремонт компьютер ноутбуков/помощник мас...,1,0
375,сервисный инженер робототехника,0,1


In [118]:
errors.shape[0]

57

В целом на правду похоже, но ошибки есть

Попробуем пропустить через модель наши данные и сохраним пресказания

In [105]:
y_pred = pipeline.predict(df['title'])

In [109]:
df['from_it'] = y_pred

In [114]:
df_trunc = df.drop("is_anomaly", axis=1)

In [115]:
df_trunc[df_trunc['from_it'] == 1].to_excel("data/it_vacancies.xlsx", index=False)

In [116]:
df_trunc[df_trunc['from_it'] == 0].to_excel("data/not_it_vacancies.xlsx", index=False)

Базовую фильтрацию мы сделали, но очень вероятно, что мы и убрали IT вакансии тоже

С помощью ИИ-агента вытянули еще IT-вакансии

In [124]:
it_vacs = pd.read_excel("data/it_vacancies_extracted.xlsx", sheet_name=0)

In [125]:
it_vacs = it_vacs.drop(["source_excel_row", "from_it", "classification_reason", "classification_confidence"], axis=1)

In [126]:
it_vacs_pt2 = pd.read_excel("data/it_vacancies_extracted.xlsx", sheet_name=1)

In [131]:
it_vacs_pt2 = it_vacs_pt2.drop(["source_excel_row", "from_it", "review_reason"], axis=1)

In [132]:
it_vacs_pt2

,title,salary_from,salary_to,salary_currency,area,experience_from,experience_to,company,skills,desc
0,Специалист по автоматизации бизнес-процессов и...,120000.0,150000.0,RUB,армавир,1.0,3.0,ООО Горница,NaN,Полная занятость\nОформление : Трудовой догово...
1,Project Manager / Менеджер проектов,118000.0,229000.0,RUB,ростов-на-дону,1.0,3.0,ФГАНУ НИИ Спецвузавтоматика,"Проектный менеджмент, Scrum, Руководство колле...",Выплаты: два раза в месяц\nПолная занятость\nО...
2,Тестировщик на производство г. Губкин,60000.0,NaN,RUB,губкин,1.0,3.0,ООО ВЭЛРОК,"Ручное тестирование, Отчетность, Диагностика н...",Выплаты: два раза в месяц\nПолная занятость\nО...
3,"Специалист по бизнес-процессам, KPI и автомати...",5000000.0,10000000.0,NaN,NaN,1.0,3.0,ООО HVARA,NaN,Выплаты: раз в месяц\nПолная занятость\nОформл...
4,Online Academic Support (Ассистент учителя),7500000.0,11000000.0,NaN,ташкент,NaN,NaN,INTER NATION ENGLISH SCHOOL (OOO INTER NATION),"Английский язык, Обучение и развитие, Точность...",Полная занятость\nГрафик: 6/1\nРабочие часы: 6...
...,...,...,...,...,...,...,...,...,...,...
230,Вакансии для веб-программиста в России,180000.0,220000.0,RUB,не указан,NaN,NaN,КОММЕРСАНТЪ,Не указаны,Не указано\nКОММЕРСАНТЪ
231,Вакансии для инженера технической поддержки в ...,61000.0,70000.0,RUB,не указан,NaN,NaN,NaN,Не указаны,Опыт работы в технической поддержке или систем...
232,Вакансии для инженера программиста в России,80000.0,NaN,RUB,не указан,NaN,NaN,ПремьерСтройДизайн,Не указаны,Не указано\nПремьерСтройДизайн
233,Консультант по автоматизации бюджетирования Пл...,116400.0,174600.0,RUB,москва,3.0,NaN,ФГУП Росморпорт,"SQL, Бюджетирование, Финансовое планирование, ...",Опыт работы от 3 лет\nОбязанности:\nАдминистри...


In [133]:
it_vacs_all = pd.concat(
    [
        df_trunc.loc[df_trunc["from_it"].eq(1)].copy(),
        it_vacs.assign(from_it=1),
        it_vacs_pt2.assign(from_it=1),
    ],
    ignore_index=True,
)

In [135]:
it_vacs_all.drop("from_it", axis=1).to_csv("data/it_vacancies_all.csv", index=False, encoding='utf-16')

In [138]:
it_vacs_all.drop("from_it", axis=1).to_excel("data/it_vacancies_all.xlsx", index=False)